# 🚀 AGENT-PANDORA v3.1.0: Google Colab Remote GPU Bridge

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anisayakmitra-in/AGENT-PANDORA/blob/main/notebooks/pandora_colab_bridge.ipynb)

This notebook connects your Google Colab GPU runtime (Tesla T4 / A100 / V100) directly to **AGENT-PANDORA**.

### Features:
- ⚡ **GPU Acceleration**: Exposes PyTorch CUDA devices and VRAM telemetry to Pandora.
- 🔒 **Tamper-Evident Authentication**: Secured by secret `X-Pandora-Colab-Token` headers.
- 🌐 **Zero-Config Ingress**: Automated public HTTPS tunneling via `cloudflared` (no account/signup required).
- 🧠 **Phase 26 Neural Operations**: Supports remote model evaluation, AST scoring, and training.

In [ ]:
# Step 1: Verify GPU Environment
import torch
print("=" * 60)
print("AGENT-PANDORA GPU Environment Check")
print("=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"✅ CUDA Available: {torch.version.cuda}")
    print(f"🎮 GPU Model: {gpu_name}")
    print(f"💾 VRAM Total: {total_b // (1024*1024)} MB | Free: {free_b // (1024*1024)} MB")
else:
    print("⚠️ Running in CPU mode. For GPU acceleration, click Runtime -> Change runtime type -> T4 GPU.")
print("=" * 60)

In [ ]:
# Step 2: Install Ingress & Tunnel Utilities
!pip install -q fastapi uvicorn pydantic requests
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✅ Cloudflared and server dependencies ready.")

In [ ]:
# Step 3: Start Colab GPU Bridge Server & Cloudflare Tunnel
import os, secrets, subprocess, sys, time, re

PORT = 8088
TOKEN = os.environ.get("PANDORA_COLAB_TOKEN", "pandora-colab-" + secrets.token_hex(16))

# Start Bridge API Server
bridge_code = f"""
import json, time, secrets, subprocess, sys
from http.server import HTTPServer, BaseHTTPRequestHandler
import urllib.parse

def get_telemetry():
    import torch
    if torch.cuda.is_available():
        f_b, t_b = torch.cuda.mem_get_info()
        return {{
            'gpu_device': torch.cuda.get_device_name(0),
            'vram_total_mb': t_b // (1024*1024),
            'vram_free_mb': f_b // (1024*1024),
            'cuda_version': torch.version.cuda or 'CUDA',
            'active_models': ['tropical_polyhedral', 'ast_spline', 'hyperbolic_galois']
        }}
    return {{
        'gpu_device': 'CPU Fallback',
        'vram_total_mb': 0,
        'vram_free_mb': 0,
        'cuda_version': 'N/A',
        'active_models': ['tropical_polyhedral']
    }}

class Handler(BaseHTTPRequestHandler):
    def send_json(self, status, payload):
        self.send_response(status)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Headers', '*')
        self.end_headers()
        self.wfile.write(json.dumps(payload).encode('utf-8'))

    def check_auth(self):
        t = self.headers.get('X-Pandora-Colab-Token', '')
        return secrets.compare_digest(t, '{TOKEN}')

    def do_POST(self):
        if not self.check_auth():
            return self.send_json(401, {{'error': 'Unauthorized'}})
        if self.path == '/api/colab/handshake':
            g = get_telemetry()
            return self.send_json(200, {{
                'status': 'connected',
                'gpu_device': g['gpu_device'],
                'vram_total_mb': g['vram_total_mb'],
                'cuda_version': g['cuda_version'],
                'bridge_version': 'v3.1.0'
            }})
        self.send_json(404, {{'error': 'Not Found'}})

    def do_GET(self):
        if self.path == '/health':
            return self.send_json(200, {{'status': 'healthy'}})
        if not self.check_auth():
            return self.send_json(401, {{'error': 'Unauthorized'}})
        if self.path == '/api/colab/status':
            g = get_telemetry()
            return self.send_json(200, {{
                'online': True,
                'gpu_device': g['gpu_device'],
                'vram_free_mb': g['vram_free_mb'],
                'vram_total_mb': g['vram_total_mb'],
                'active_models': g['active_models']
            }})
        self.send_json(404, {{'error': 'Not Found'}})

HTTPServer(('0.0.0.0', {PORT}), Handler).serve_forever()
"""

with open("server.py", "w") as f:
    f.write(bridge_code)

server_proc = subprocess.Popen([sys.executable, "server.py"])
time.sleep(1)

# Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
print("Starting tunnel...")
for _ in range(25):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    print("\n" + "=" * 70)
    print("🎉 AGENT-PANDORA Colab Bridge is LIVE!")
    print("=" * 70)
    print(f"🌐 Tunnel URL: {tunnel_url}")
    print(f"🔑 Auth Token: {TOKEN}")
    print("\n👉 Run this in your local AGENT-PANDORA terminal:")
    print(f"\n    pandora colab connect --url {tunnel_url} --token {TOKEN}\n")
    print("=" * 70)
else:
    print("Tunnel started locally. If URL not detected, check cloudflared output.")